In [ ]:
from pyspark.sql.functions import count,col, current_timestamp, round, sum as spark_sum
from delta.tables import DeltaTable

In [ ]:
storage_key = dbutils.secrets.get(
    scope="your-secret-scope",
    key="azure-storage-key"
)

spark.conf.set(
    "fs.azure.account.key.stgrocerywaste.dfs.core.windows.net",
    storage_key
)


In [ ]:
landing_path = "abfss://landing@stgrocerywaste.dfs.core.windows.net/gro_incremental.csv"

bronze_path = "abfss://bronze@stgrocerywaste.dfs.core.windows.net/inventory"
silver_path = "abfss://silver@stgrocerywaste.dfs.core.windows.net/inventory"
gold_path = "abfss://gold@stgrocerywaste.dfs.core.windows.net/analytics"

In [ ]:
df_inc = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load(landing_path)

In [ ]:
df_inc = df_inc.withColumn("ingestion_time", current_timestamp())

In [ ]:
df_inc.write \
.mode("append") \
.format("delta") \
.save(bronze_path)

In [ ]:
df_bronze = spark.read.format("delta").load(bronze_path)

In [ ]:
df_silver = df_bronze \
.dropDuplicates() \
.dropna(subset=["Transaction_ID", "Store_ID", "Product_Name"]) \
.withColumn("Stock_Date", col("Stock_Date").cast("date")) \
.withColumn("Expiry_Date", col("Expiry_Date").cast("date")) \
.withColumn("Scan_Date", col("Scan_Date").cast("date"))

In [ ]:
df_silver.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.format("delta") \
.save(silver_path)

In [ ]:
df = spark.read.format("delta").load(silver_path)

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round

store_waste = df.withColumn(
    "Waste_Cost",
    col("Quantity_Received") * col("Unit_Cost")
).groupBy("Store_ID") \
.agg(
    round(spark_sum("Waste_Cost"), 2).alias("Total_Waste_Cost"),
    spark_sum("Quantity_Received").alias("Total_Received")
)

store_waste.show(truncate=False)

In [ ]:
category_waste = df.withColumn(
    "Waste_Cost",
    col("Quantity_Received") * col("Unit_Cost")
).groupBy("Product_Category") \
.agg(
    round(spark_sum("Waste_Cost"), 2).alias("Total_Waste_Cost"),
    count("*").alias("Total_Records")
)

category_waste.show(truncate=False)

In [ ]:
product_waste = df.withColumn(
    "Waste_Cost",
    col("Quantity_Received") * col("Unit_Cost")
).groupBy("Product_Name") \
.agg(
    round(spark_sum("Waste_Cost"), 2).alias("Total_Waste_Cost")
)

product_waste.show(truncate=False)

In [ ]:
discount_impact = df.groupBy("Discount_Applied") \
.agg(
    round(spark_sum("Quantity_Received"), 2).alias("Total_Received"),
    count("*").alias("Transactions")
)

discount_impact.show(truncate=False)

In [ ]:
from pyspark.sql.functions import datediff

high_risk = df.withColumn(
    "Days_To_Expire",
    datediff(col("Expiry_Date"), col("Scan_Date"))
).filter(col("Days_To_Expire") <= 3) \
.groupBy("Product_Name") \
.agg(
    count("*").alias("High_Risk_Count")
)

high_risk.show(truncate=False)